In [ ]:
import os
os.environ['http_proxy'] = 'http://127.0.0.1:7890'
os.environ['https_proxy'] = 'http://127.0.0.1:7890'
os.environ['all_proxy'] = 'socks5://127.0.0.1:7890'

os.environ['HF_HOME'] = '/home/bobsun/cambrige/MedMinist/hugginface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/home/bobsun/cambrige/MedMinist/hugginface'
from datasets import load_dataset

# 加载 AG News 数据集
dataset = load_dataset('ag_news')

In [1]:
from transformers import BertTokenizer
# 加载 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess_function(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=70)
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': examples['label']
    }

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=["text"])


NameError: name 'dataset' is not defined

In [3]:
# 加载数据集
# 计算文本长度
def calculate_length(example):
    return {"length": len(tokenizer(example["text"])["input_ids"])}

# 对训练集统计文本长度
length_stats = dataset['train'].map(calculate_length)
lengths = length_stats["length"]

# 打印统计信息
print(f"最大长度: {max(lengths)}")
print(f"平均长度: {sum(lengths)/len(lengths)}")
print(f"90% 样本长度: {sorted(lengths)[int(0.9 * len(lengths))]}")

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

最大长度: 379
平均长度: 53.16641666666667
90% 样本长度: 70


In [5]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

# 定义 DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='max_length', max_length=70, return_tensors='pt')

# 创建 DataLoader，指定 collate_fn
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=data_collator)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=data_collator)


In [8]:
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
import torch
from tqdm import tqdm
from torch.optim import AdamW
from transformers import logging
import warnings
warnings.filterwarnings("ignore", message="Some weights of BertForSequenceClassification were not initialized from the model checkpoint")

# 加载预训练的 BERT 模型
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)
logging.set_verbosity_error()
# 如果有可用的 GPU，请将模型转移到 GPU 上
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# 使用 AdamW 优化器
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# 定义学习率调度器
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# 定义损失函数（已在模型内部实现）
# 损失已经由 BertForSequenceClassification 计算

for epoch in range(epochs):
    model.train()  # 切换到训练模式
    total_loss = 0

    # 使用 tqdm 显示训练进度条
    for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch +1}"):
        # 获取输入数据和标签
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # 清除之前的梯度
        optimizer.zero_grad()

        # 前向传播
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss  # 获取损失

        # 反向传播
        loss.backward()

        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 优化器更新参数
        optimizer.step()

        # 更新学习率
        scheduler.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch +1}, Average Training Loss: {avg_train_loss:.4f}")

    # 验证步骤
    model.eval()
    correct = 0
    total = 0
    eval_loss = 0
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc=f"Validation Epoch {epoch +1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            eval_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_eval_loss = eval_loss / len(test_dataloader)
    accuracy = correct / total
    print(f"Epoch {epoch +1}, Validation Loss: {avg_eval_loss:.4f}, Validation Accuracy: {accuracy * 100:.2f}%")


Training Epoch 1: 100%|██████████| 3750/3750 [03:11<00:00, 19.63it/s]


Epoch 1, Average Training Loss: 0.2192


Validation Epoch 1: 100%|██████████| 238/238 [00:03<00:00, 71.98it/s]


Epoch 1, Validation Loss: 0.1731, Validation Accuracy: 94.29%


Training Epoch 2: 100%|██████████| 3750/3750 [03:10<00:00, 19.64it/s]


...

Epoch 2, Validation Loss: 0.1756, Validation Accuracy: 94.41%


Training Epoch 3: 100%|██████████| 3750/3750 [03:10<00:00, 19.64it/s]


Epoch 3, Average Training Loss: 0.0818


Validation Epoch 3: 100%|██████████| 238/238 [00:03<00:00, 72.05it/s]

Epoch 3, Validation Loss: 0.1903, Validation Accuracy: 94.61%


In [10]:
torch.save(model.state_dict(), 'bert_ag_news.pth')

In [11]:
for i, layer in enumerate(model.bert.encoder.layer):
    print(model.bert.encoder.layer[i])


BertLayer(
  (attention): BertAttention(
    (self): BertSdpaSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)
BertLayer(
  (attention): BertAttention(
    (self): BertSdpaSelfAttention(
 

In [15]:
for i, layer in enumerate(model.bert.encoder.layer):
    print(f"Layer {i + 1}:")
    print(layer)

Layer 1:
BertLayer(
  (attention): BertAttention(
    (self): BertSdpaSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)
Layer 2:
BertLayer(
  (attention): BertAttention(
    (self): BertSd